[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_26/2_sql.ipynb)

# Day 26: SQL (hard)

**Company style: Google.** Search, ads and Play Store data. Google SQL rounds like clean definitions (what is a 'good click'?), compressed tables, and edge cases such as ties and duplicates. Say how you would validate the result.

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (30 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Median from a compressed histogram | hard | 6 |
| Q2 | Queries where users do not find what they need | hard | 6 |
| Q3 | How often do users rephrase? | hard | 7 |
| Q4 | Top apps per category on Google Play (review) | hard | 5 |
| Q5 | When did each advertiser hit the budget? (review) | hard | 6 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Made-up search tables** (invented for practice, fixed seed; one week from 2024-06-03):

| Table | One row is | Columns |
|---|---|---|
| `searches` | one query typed by a user | `search_id`, `user_id`, `search_time` ('YYYY-MM-DD HH:MM:SS'), `query`, `device` (mobile / desktop / tablet) |
| `clicks` | one click on a result | `search_id`, `position` (1 = top result), `click_time`, `dwell_s` (seconds on the page before coming back; > 30 counts as a "good click") |

**Real table: Google Play Store** (scraped in 2018, Kaggle "lava18/google-play-store-apps" mirror; study use only):

| Table | One row is | Columns |
|---|---|---|
| `play_apps` | one scraped listing (10,840). Some apps appear more than once (scraped twice) | `app`, `category` (e.g. 'GAME'), `rating` (1 to 5, NULL if unrated), `reviews`, `installs` (lower bound of the bucket, e.g. 1000000 for '1,000,000+'), `type` (Free / Paid), `price` (USD), `content_rating`, `genres`, `last_updated` ('YYYY-MM-DD') |

**Made-up small tables:** `search_freq` (`searches_per_week`, `num_users`): a compressed histogram, e.g. the row
(2, 120) means 120 users searched exactly 2 times. `ad_budgets` (`advertiser_id`, `budget`) and `ad_spend`
(`advertiser_id`, `spend_date`, `spend`): one row per spend record; a day can have several records.

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up web search log (generated with a fixed seed).
import numpy as np

def make_search_tables(con, seed=26):
    rs = np.random.RandomState(seed)
    topics = ["weather", "flights to seoul", "python list sort", "pizza near me", "nba scores", "tax deadline",
              "how to tie a tie", "galaxy s24 review", "bitcoin price", "sql window functions", "cheap hotels paris",
              "translate hello", "movie times", "covid symptoms", "best laptop 2024"]
    quality = rs.beta(4, 2, len(topics))                      # how often result 1 answers the query
    quality[[2, 9, 14]] = [0.25, 0.30, 0.20]                   # three hard queries
    devices = ["mobile", "desktop", "tablet"]
    start = pd.Timestamp("2024-06-03")
    s_rows, c_rows = [], []
    sid = 0
    for u in range(1, 801):
        dev = devices[rs.choice(3, p=[.6, .32, .08])]
        for _ in range(rs.poisson(5)):
            t = int(rs.randint(0, 7 * 86400))
            k = rs.randint(len(topics))
            q = topics[k]
            for attempt in range(3):
                sid += 1
                s_rows.append((sid, u, start + pd.Timedelta(seconds=t), q, dev))
                p_ok = quality[k] if attempt == 0 else min(quality[k] + 0.3, 0.95)
                if rs.rand() < p_ok:                           # satisfied: one long click near the top
                    pos = 1 if rs.rand() < 0.7 else int(rs.randint(2, 4))
                    c_rows.append((sid, pos, start + pd.Timedelta(seconds=t + int(rs.randint(2, 12))), int(rs.randint(31, 400))))
                    break
                if rs.rand() < 0.5:                            # short click, then back to the results
                    pos = int(rs.randint(1, 6))
                    c_rows.append((sid, pos, start + pd.Timedelta(seconds=t + int(rs.randint(2, 12))), int(rs.randint(2, 30))))
                    t += int(rs.randint(20, 60))
                if rs.rand() < 0.55:                           # reformulate the query a few seconds later
                    t += int(rs.randint(5, 50))
                    q = q + (" " + ["2024", "best", "how", "free", "near me"][rs.randint(5)])
                else:
                    break
    fmt = "%Y-%m-%d %H:%M:%S"
    s = pd.DataFrame(s_rows, columns=["search_id", "user_id", "search_time", "query", "device"])
    s["search_time"] = s["search_time"].dt.strftime(fmt)
    c = pd.DataFrame(c_rows, columns=["search_id", "position", "click_time", "dwell_s"])
    c["click_time"] = c["click_time"].dt.strftime(fmt)
    s.to_sql("searches", con, index=False, if_exists="replace")
    c.to_sql("clicks", con, index=False, if_exists="replace")

make_search_tables(db)

# Real data: Google Play Store apps (scraped 2018, Kaggle lava18 mirror; study use only). Cleaned into numbers.
_a = pd.read_csv(data_path("googleplaystore.csv", "https://raw.githubusercontent.com/malborroni/Foundations_of_Computer-Science/master/datasets/googleplaystore.csv"))
_a = _a[_a["Category"] != "1.9"].copy()          # one shifted row in the raw file
_a["App"] = _a["App"].str.replace("[\u2013\u2014]", "-", regex=True)   # long dashes in names -> '-'
_a["reviews"] = pd.to_numeric(_a["Reviews"], errors="coerce").astype("Int64")
_a["installs"] = pd.to_numeric(_a["Installs"].str.replace("[+,]", "", regex=True), errors="coerce").astype("Int64")
_a["price"] = pd.to_numeric(_a["Price"].str.replace("$", "", regex=False), errors="coerce")
_a["last_updated"] = pd.to_datetime(_a["Last Updated"], format="%B %d, %Y").dt.strftime("%Y-%m-%d")
_a = _a.rename(columns={"App": "app", "Category": "category", "Rating": "rating", "Type": "type",
                        "Content Rating": "content_rating", "Genres": "genres"})
_a[["app", "category", "rating", "reviews", "installs", "type", "price", "content_rating", "genres",
    "last_updated"]].to_sql("play_apps", db, index=False)

# Made-up small tables for classic Google-style questions.
db.executescript("""
CREATE TABLE search_freq (searches_per_week INTEGER, num_users INTEGER);
INSERT INTO search_freq VALUES (0, 120), (1, 260), (2, 120), (3, 230), (4, 140), (5, 60), (7, 40), (10, 20), (20, 10);
CREATE TABLE ad_budgets (advertiser_id INTEGER, budget REAL);
INSERT INTO ad_budgets VALUES (1, 1000), (2, 500), (3, 2000), (4, 300);
CREATE TABLE ad_spend (advertiser_id INTEGER, spend_date TEXT, spend REAL);
INSERT INTO ad_spend VALUES
(1, '2024-06-03', 220), (1, '2024-06-04', 310), (1, '2024-06-05', 180), (1, '2024-06-06', 400), (1, '2024-06-07', 90),
(2, '2024-06-03', 150), (2, '2024-06-04', 150), (2, '2024-06-05', 200), (2, '2024-06-06', 120),
(3, '2024-06-03', 300), (3, '2024-06-05', 450), (3, '2024-06-06', 500), (3, '2024-06-07', 380),
(4, '2024-06-04', 120), (4, '2024-06-04', 90), (4, '2024-06-06', 150);
""")

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Median from a compressed histogram

*hard, about 6 min*

`search_freq` stores how many users searched N times last week (one row per N) instead of one row per
user. Return the **median** searches per week (1 decimal) without expanding the table into one row per user.

Example: rows (1, 2), (5, 1), (9, 1) mean the values 1, 1, 5, 9: the median is (1 + 5) / 2 = 3.0.

*Follow-up:* how would you get p90 with the same idea?

*Source: [DataLemur Google SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/google-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: a frequency table (value, count) and a percentile. Pattern: running totals of the counts give each value's position range; the median is the value whose range contains the middle position(s).

</details>

<details><summary><b>Hint 2</b></summary>

1. `SUM(num_users) OVER (ORDER BY searches_per_week ROWS UNBOUNDED PRECEDING)` = last position of each value.
2. First position = that minus num_users plus 1; n = SUM(num_users) OVER ().
3. Middle positions are `(n + 1) / 2` and `(n + 2) / 2` (integer division). AVG the values whose range contains either.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH c AS (
  SELECT searches_per_week AS x, num_users,
    SUM(num_users) OVER (ORDER BY searches_per_week ROWS UNBOUNDED PRECEDING) AS last_pos,
    SUM(num_users) OVER () AS n
  FROM search_freq
), r AS (
  SELECT x, last_pos - num_users + 1 AS first_pos, last_pos, n FROM c
)
SELECT ROUND(AVG(x), 1) AS median
FROM r
WHERE (n + 1) / 2 BETWEEN first_pos AND last_pos
   OR (n + 2) / 2 BETWEEN first_pos AND last_pos
```

Result:

| median |
|---|
| 2.5 |

**Why:** There are 1,000 users, so the median averages positions 500 and 501. The running total reaches exactly 500 at value 2 (120 + 260 + 120), so position 500 is a 2 and position 501 is a 3: median 2.5. Many solutions return 2 or 3 because they only look for the first value whose running share passes 50%. The same range logic works for odd n (both expressions give the same position). Follow-up: p90 (nearest rank) is the value whose range contains position `CEIL(0.9 * n)` = 900: value 5 (positions 871 to 930).

**Complexity:** One window over the distinct values: O(k log k) for k rows, no matter how many users.

**Common mistakes:** Expanding with a recursive CTE (works, but n rows; mention it as the slow alternative). Default RANGE frame (fine here because values are unique, but say ROWS). Picking only one middle value.

**Learn more:** [sql-median-percentiles](https://mahsa-agz.github.io/ds-handbook/#sql-median-percentiles), [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals)

</details>

---
### Q2. Queries where users do not find what they need

*hard, about 6 min*

A search has a **good click** if any of its clicks has `dwell_s > 30`. For every exact `query` text with at
least 200 searches return `searches`, `ctr_pct` (searches with any click), `good_pct` (searches with a good click) and
`pos1_pct` (searches with a click on position 1), all in % with 1 decimal. Show the 5 queries with the lowest `good_pct`.

*Follow-up:* CTR is above 60% for all of them. Why is good-click rate the better quality metric?

*Source: [DataLemur Google SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/google-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: rates per query where a search can have 0, 1 or many clicks. Pattern: ratios: collapse to one row per search with MAX(CASE ...) flags, then AVG the flags per query.

</details>

<details><summary><b>Hint 2</b></summary>

1. `searches LEFT JOIN clicks`, GROUP BY search: `MAX(CASE WHEN c.dwell_s > 30 THEN 1 ELSE 0 END)` etc.
2. GROUP BY query HAVING COUNT(*) >= 200: `100.0 * AVG(flag)`.
3. ORDER BY good_pct LIMIT 5.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH per_search AS (
  SELECT s.search_id, s.query,
    MAX(CASE WHEN c.search_id IS NOT NULL THEN 1 ELSE 0 END) AS any_click,
    MAX(CASE WHEN c.dwell_s > 30 THEN 1 ELSE 0 END) AS good_click,
    MAX(CASE WHEN c.position = 1 THEN 1 ELSE 0 END) AS pos1_click
  FROM searches s
  LEFT JOIN clicks c ON c.search_id = s.search_id
  GROUP BY s.search_id, s.query
)
SELECT query, COUNT(*) AS searches,
  ROUND(100.0 * AVG(any_click), 1) AS ctr_pct,
  ROUND(100.0 * AVG(good_click), 1) AS good_pct,
  ROUND(100.0 * AVG(pos1_click), 1) AS pos1_pct
FROM per_search
GROUP BY query
HAVING COUNT(*) >= 200
ORDER BY good_pct
LIMIT 5
```

Result:

| query | searches | ctr_pct | good_pct | pos1_pct |
|---|---|---|---|---|
| nba scores | 263 | 62.7 | 25.9 | 26.6 |
| best laptop 2024 | 271 | 63.1 | 26.6 | 26.9 |
| python list sort | 260 | 64.2 | 27.3 | 27.3 |
| sql window functions | 254 | 64.2 | 28.0 | 26.0 |
| galaxy s24 review | 253 | 78.3 | 59.7 | 46.6 |

**Why:** Collapsing to one row per search first is the key step: after the LEFT JOIN a search with two clicks has two rows, and averaging those rows would weight it twice. 'nba scores' has 62.7% CTR but only 25.9% good clicks. Follow-up: a short click followed by a quick return (pogo-sticking) is a click, but a failure; good-click (long dwell) rate tracks satisfaction, so queries with high CTR but low good-click rate are the ones to send to the ranking team.

**Complexity:** One join and two GROUP BYs: O(searches + clicks).

**Common mistakes:** AVG over the joined rows (multi-click searches count twice). `COUNT(c.search_id) / COUNT(*)` (clicks per search, not a rate). Filtering `WHERE dwell_s > 30` before the join (drops searches without good clicks from the denominator).

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls)

</details>

---
### Q3. How often do users rephrase?

*hard, about 7 min*

A search is **abandoned and rephrased** if it has no good click (`dwell_s > 30`) AND the same user runs
their next search within 120 seconds. Return per `device`: `searches` and `reform_pct` (% of all searches that were
abandoned and rephrased, 1 decimal). Order by device.

Example: user 7 searches 'weather' at 10:00:00, short click (5 s dwell), searches 'weather 2024' at 10:00:40: the
first search counts. The second search counts only if it is also followed within 120 s.

*Source: [DataLemur Google SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/google-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: compare each search with the user's NEXT search. Pattern: LEAD over the user's searches, after collapsing clicks to one row per search.

</details>

<details><summary><b>Hint 2</b></summary>

1. One row per search: MAX(CASE good click) from the LEFT JOIN, GROUP BY search.
2. In the same SELECT (windows run after GROUP BY): `LEAD(search_time) OVER (PARTITION BY user_id ORDER BY search_time, search_id)`.
3. Flag = no good click AND next time - this time <= 120 s; AVG per device.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH g AS (
  SELECT s.search_id, s.user_id, s.device, s.search_time,
    MAX(CASE WHEN c.dwell_s > 30 THEN 1 ELSE 0 END) AS good_click,
    LEAD(s.search_time) OVER (PARTITION BY s.user_id ORDER BY s.search_time, s.search_id) AS next_time
  FROM searches s
  LEFT JOIN clicks c ON c.search_id = s.search_id
  GROUP BY s.search_id, s.user_id, s.device, s.search_time
)
SELECT device, COUNT(*) AS searches,
  ROUND(100.0 * AVG(CASE WHEN good_click = 0
      AND (julianday(next_time) - julianday(search_time)) * 86400 <= 120
    THEN 1 ELSE 0 END), 1) AS reform_pct
FROM g
GROUP BY device
ORDER BY device
```

Result:

| device | searches | reform_pct |
|---|---|---|
| desktop | 1568 | 18.0 |
| mobile | 2896 | 19.3 |
| tablet | 417 | 19.9 |

**Why:** Window functions are evaluated after GROUP BY, so LEAD here walks over one row per search, not one per click. When there is no next search, `next_time` is NULL, the comparison is NULL, and the CASE gives 0. About 18% to 20% of searches on every device end in a quick rephrase; that is a direct 'search failed' signal used as a guardrail metric in ranking experiments.

**Complexity:** One join, one GROUP BY, one window sort per user: O(n log n).

**Common mistakes:** LEAD over the raw joined rows (the 'next' row can be another click of the same search). Partitioning by query instead of user (a rephrase changes the query text). Not ordering by a tiebreaker (search_id) when two searches share a second.

**Learn more:** [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
### Q4. Top apps per category on Google Play (review)

*hard, about 5 min*

Google Play (real, scraped). Some apps appear several times in the same category (scraped twice, with
slightly different review counts). Keep one row per (app, category): the one with the most reviews. Then return the
**top 3 apps per category** by `installs`, breaking ties by `reviews`, for the categories COMMUNICATION, GAME and
PHOTOGRAPHY: `category`, `rn`, `app`, `installs`, `reviews`.

*Follow-up:* `installs` is a bucket (1,000,000,000+ means "at least a billion"). What does that do to a top-N by installs?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: duplicates, then 'top 3 per group' with ties. Pattern: dedup with ROW_NUMBER, then top-N per group with a second ROW_NUMBER and a tiebreaker.

</details>

<details><summary><b>Hint 2</b></summary>

1. `ROW_NUMBER() OVER (PARTITION BY app, category ORDER BY reviews DESC)`, keep 1.
2. `ROW_NUMBER() OVER (PARTITION BY category ORDER BY installs DESC, reviews DESC)`, keep <= 3.
3. Filter the 3 categories at the end (or before ranking: same result here).

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH one AS (
  SELECT app, category, installs, reviews,
    ROW_NUMBER() OVER (PARTITION BY app, category ORDER BY reviews DESC) AS dup
  FROM play_apps
), ranked AS (
  SELECT app, category, installs, reviews,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY installs DESC, reviews DESC) AS rn
  FROM one
  WHERE dup = 1
)
SELECT category, rn, app, installs, reviews
FROM ranked
WHERE category IN ('COMMUNICATION', 'GAME', 'PHOTOGRAPHY') AND rn <= 3
ORDER BY category, rn
```

Result:

| category | rn | app | installs | reviews |
|---|---|---|---|---|
| COMMUNICATION | 1 | WhatsApp Messenger | 1000000000 | 69119316 |
| COMMUNICATION | 2 | Messenger - Text and Video Chat for Free | 1000000000 | 56646578 |
| COMMUNICATION | 3 | Skype - free IM & video calls | 1000000000 | 10484169 |
| GAME | 1 | Subway Surfers | 1000000000 | 27725352 |
| GAME | 2 | Candy Crush Saga | 500000000 | 22430188 |
| GAME | 3 | My Talking Tom | 500000000 | 14892469 |
| PHOTOGRAPHY | 1 | Google Photos | 1000000000 | 10859051 |
| PHOTOGRAPHY | 2 | PicsArt Photo Studio: Collage Maker & Pic Editor | 100000000 | 7594559 |
| PHOTOGRAPHY | 3 | PhotoGrid: Video & Pic Collage Maker, Photo Editor | 100000000 | 7529865 |

**Why:** Without the dedup step, WhatsApp Messenger takes ranks 1, 2 and 3 of COMMUNICATION (three scraped copies), and the same happens to Subway Surfers and Google Photos. 10,840 rows hold only 9,744 distinct (app, category) pairs. Follow-up: many apps share the top bucket (three COMMUNICATION apps have 1,000,000,000), so the order inside a bucket comes from the tiebreaker (reviews). Say that the ranking is coarse, and use RANK or DENSE_RANK if ties should share a place.

**Complexity:** Two window sorts: O(n log n).

**Common mistakes:** SELECT DISTINCT (the copies differ in reviews, so they are not exact duplicates). RANK without a tiebreaker (returns 4+ rows when the 3rd place is tied, which may or may not be what is asked: say it).

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning)

</details>

---
### Q5. When did each advertiser hit the budget? (review)

*hard, about 6 min*

Each advertiser has a total `budget` (`ad_budgets`). Spend arrives as records in `ad_spend` (a day can have
several records). For **every** advertiser return `advertiser_id`, `budget`, `reached_on` (the first day the
cumulative spend is >= budget), `cum_spend` on that day and `over_by` (cum_spend - budget). Advertisers who never
reach the budget get NULLs. Order by advertiser.

Example: budget 500, daily spend 150, 150, 200: cumulative 150, 300, 500: reached on day 3, over_by 0.

*Source: [DataLemur Google SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/google-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'first day the cumulative total crosses X'. Pattern: running total with SUM OVER (ORDER BY day), then MIN(day) where the total >= budget, LEFT JOINed back to all advertisers.

</details>

<details><summary><b>Hint 2</b></summary>

1. `daily`: SUM(spend) per advertiser and day (several records per day).
2. `run`: `SUM(spend) OVER (PARTITION BY advertiser_id ORDER BY spend_date ROWS UNBOUNDED PRECEDING)`.
3. `hit`: MIN(spend_date) WHERE cum >= budget.
4. `ad_budgets LEFT JOIN hit LEFT JOIN run` on the hit day.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH daily AS (
  SELECT advertiser_id, spend_date, SUM(spend) AS spend
  FROM ad_spend
  GROUP BY advertiser_id, spend_date
), run AS (
  SELECT advertiser_id, spend_date,
    SUM(spend) OVER (PARTITION BY advertiser_id ORDER BY spend_date
                     ROWS UNBOUNDED PRECEDING) AS cum
  FROM daily
), hit AS (
  SELECT r.advertiser_id, MIN(r.spend_date) AS reached_on
  FROM run r
  JOIN ad_budgets b ON b.advertiser_id = r.advertiser_id
  WHERE r.cum >= b.budget
  GROUP BY r.advertiser_id
)
SELECT b.advertiser_id, b.budget, h.reached_on,
  r.cum AS cum_spend, r.cum - b.budget AS over_by
FROM ad_budgets b
LEFT JOIN hit h ON h.advertiser_id = b.advertiser_id
LEFT JOIN run r ON r.advertiser_id = b.advertiser_id AND r.spend_date = h.reached_on
ORDER BY b.advertiser_id
```

Result:

| advertiser_id | budget | reached_on | cum_spend | over_by |
|---|---|---|---|---|
| 1 | 1000.0 | 2024-06-06 | 1110.0 | 110.0 |
| 2 | 500.0 | 2024-06-05 | 500.0 | 0.0 |
| 3 | 2000.0 | NULL | NULL | NULL |
| 4 | 300.0 | 2024-06-06 | 360.0 | 60.0 |

**Why:** Aggregating to one row per day first makes the running total and the join on the hit day unique (advertiser 4 has two records on 2024-06-04). Advertiser 2 reaches exactly 500 on 2024-06-05 (over_by 0, so `>=` matters); advertiser 3 spends 1,630 of 2,000 and correctly gets NULLs thanks to the LEFT JOINs. In production this is the query behind 'pacing' alerts: the over_by column is money the advertiser did not want to spend.

**Complexity:** One aggregation and one window sort per advertiser.

**Common mistakes:** Running total over raw records joined back by date (duplicate rows for advertiser 4). `>` instead of `>=`. Inner joins that drop advertiser 3. RANGE frame with duplicate dates (both records of a day get the day's total, which hides the order inside the day).

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_26/3_stats.ipynb)